## Open notebook in:
| Colab                                 |  Gradient    [link text](https://)                                                                                                                                     |
|:-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|:-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nicolepcx/transformers-the-definitive-guide/blob/master/CH04/CH04_patch_tst_hyperparameter_IBM_10_days_ahead_32_context_window.ipynb)                                             | [![Gradient](https://assets.paperspace.io/img/gradient-badge.svg)](https://console.paperspace.com//github.com/Nicolepcx/transformers-the-definitive-guide/blob/main/CH02/ch02_patch_tst_hyperparameter_IBM_10_days_ahead_32_context_window.ipynb)|             

# About this Notebook

This notebook provides an overview on how to use the `PatchTST` model from the Hugging Face `transformers` library. The workflow involves loading and preprocessing time series data, configuring the model, performing hyperparameter tuning with Optuna, and finally training and evaluating the model.

### Steps Included:

1. **Setting Up the Environment**:  
   Import necessary libraries, suppress warnings, and set a random seed to ensure reproducibility.

2. **Loading and Preparing the Dataset**:  
   Load time series data (e.g., stock prices) into a Pandas DataFrame. The data is then split into training, validation, and test sets based on specified indices. The `TimeSeriesPreprocessor` is used to normalize and prepare the data for model training.

3. **Configuring the PatchTST Model**:  
   Define the `PatchTSTConfig` configuration for the model, specifying parameters such as the number of input channels, context length, patch length, and the model's architecture. The model is initialized using this configuration.

4. **Hyperparameter Tuning with Optuna**:  
   Use Optuna to perform a hyperparameter search, exploring different configurations of learning rate, batch size, number of epochs, and other parameters. The goal is to minimize the evaluation loss, and early stopping is implemented to prevent overfitting.

5. **Training the Model**:  
   The `Trainer` class from `transformers` is used to handle the training loop. The best hyperparameters found during the Optuna search are applied to the training arguments, and the model is trained on the time series data.

6. **Model Evaluation**:  
   After training, the model is evaluated on the validation and test datasets. The results, including evaluation metrics such as loss, are printed to assess the model's performance.

7. **Saving the Model**:  
   Finally, the trained model and its configurations are saved for future use, allowing for easy deployment and inference on new time series data.

The notebook is partially based on the example code from the original [paper's repo](https://github.com/yuqinie98/PatchTST).


In [ ]:
#!pip install git+https://github.com/IBM/tsfm.git -qqq
!pip install -U git+https://github.com/ibm-granite/granite-tsfm.git -qqq

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 83.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 78.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 20.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.3 which is incompatible.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.3 which is incompatible.
bqplot 0.12.45 requires pandas<3.0.0,>=1.0.0, but you have pandas 3.0.3 which is incompatible.
db-dtypes

In [ ]:
!pip install -q optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 10.6 MB/s eta 0:00:00


In [ ]:
# Standard
import os
import numpy as np
import pandas as pd

# Third Party
from transformers import (
    EarlyStoppingCallback,
    PatchTSTConfig,
    PatchTSTForPrediction,
    set_seed,
    Trainer,
    TrainingArguments,
)
import optuna
import yfinance as yf

# First Party
from tsfm_public.toolkit.dataset import ForecastDFDataset
from tsfm_public.toolkit.time_series_preprocessor import TimeSeriesPreprocessor
from tsfm_public.toolkit.util import select_by_index

# Supress some warnings
import warnings

warnings.filterwarnings("ignore", module="torch")

### Set seed

In [ ]:
set_seed(42)

# Load and prepare datasets

In the next cell, please adjust the following parameters to suit your application:
- `PRETRAIN_AGAIN`: Set this to `True` if you want to perform pretraining again. Note that this might take some time depending on the GPU availability. Otherwise, the already pretrained model will be used.
- `dataset_path`: path to local .csv file, or web address to a csv file for the data of interest. Data is loaded with pandas, so anything supported by
`pd.read_csv` is supported: (https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.read_csv.html).
- `timestamp_column`: column name containing timestamp information, use None if there is no such column
- `id_columns`: List of column names specifying the IDs of different time series. If no ID column exists, use []
- `forecast_columns`: List of columns to be modeled
- `context_length`: The amount of historical data used as input to the model. Windows of the input time series data with length equal to
`context_length` will be extracted from the input dataframe. In the case of a multi-time series dataset, the context windows will be created
so that they are contained within a single time series (i.e., a single ID).
- `forecast_horizon`: Number of timestamps to forecast in future.
- `train_start_index`, `train_end_index`: the start and end indices in the loaded data which delineate the training data.
- `valid_start_index`, `valid_end_index`: the start and end indices in the loaded data which delineate the validation data.
- `test_start_index`, `test_end_index`: the start and end indices in the loaded data which delineate the test data.
- `patch_length`: The patch length for the `PatchTSMixer` model. It is recommended to choose a value that evenly divides `context_length`.
- `num_workers`: Number of dataloder workers in pytorch dataloader.
- `batch_size`: Batch size.
The data is first loaded into a Pandas dataframe and split into training, validation, and test parts. Then the pandas dataframes are converted
to the appropriate torch dataset needed for training.

In [ ]:
data = yf.download("IBM", start="1962-01-02", end="2024-07-01")

/tmp/ipykernel_4684/1051633117.py:1: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download("IBM", start="1962-01-02", end="2024-07-01")
/usr/local/lib/python3.12/dist-packages/yfinance/scrapers/history.py:204: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  dt_now = pd.Timestamp.utcnow()
[*********************100%***********************]  1 of 1 completed


In [ ]:
data

Price,Close,High,Low,Open,Volume
Ticker,IBM,IBM,IBM,IBM,IBM
Date,,,,,
1962-01-02,1.444739,1.461156,1.444739,1.461156,407940
1962-01-03,1.457367,1.457367,1.444738,1.444738,305955
1962-01-04,1.442844,1.457367,1.442213,1.457367,274575
1962-01-05,1.414429,1.440949,1.411903,1.440949,384405
1962-01-08,1.387909,1.413166,1.376543,1.413166,572685
...,...,...,...,...,...
2024-06-24,165.630112,168.895216,164.816204,165.620653,4864700
2024-06-25,163.349258,166.330424,162.232495,165.753117,4069800


In [ ]:
data.columns = data.columns.droplevel("Ticker")
data = data.reset_index()
data

Price,Date,Close,High,Low,Open,Volume
0,1962-01-02,1.444739,1.461156,1.444739,1.461156,407940
1,1962-01-03,1.457367,1.457367,1.444738,1.444738,305955
2,1962-01-04,1.442844,1.457367,1.442213,1.457367,274575
3,1962-01-05,1.414429,1.440949,1.411903,1.440949,384405
4,1962-01-08,1.387909,1.413166,1.376543,1.413166,572685
...,...,...,...,...,...,...
15725,2024-06-24,165.630112,168.895216,164.816204,165.620653,4864700
15726,2024-06-25,163.349258,166.330424,162.232495,165.753117,4069800
15727,2024-06-26,162.658401,163.424986,161.276660,162.100027,2779000
15728,2024-06-27,161.693069,163.254630,161.342890,161.948588,2894000


In [ ]:
data.columns.name = None

In [ ]:
data = data[["Date", 'Close']]

In [ ]:
data = data.reset_index(drop=True)

In [ ]:
data = data.rename(columns={'Close': 'close', 'Date': 'date'})
data

,date,close
0,1962-01-02,1.444739
1,1962-01-03,1.457367
2,1962-01-04,1.442844
3,1962-01-05,1.414429
4,1962-01-08,1.387909
...,...,...
15725,2024-06-24,165.630112
15726,2024-06-25,163.349258
15727,2024-06-26,162.658401
15728,2024-06-27,161.693069


In [ ]:
data.columns

Index(['date', 'close'], dtype='str')

In [ ]:
timestamp_column = "date"
id_columns = []

context_length = 32
forecast_horizon = 10
patch_length = 8
num_workers = 16
batch_size = 128

In [ ]:

forecast_columns = list(data.columns[1:])

# get split
num_train = int(len(data) * 0.7)
num_test = int(len(data) * 0.2)
num_valid = len(data) - num_train - num_test
border1s = [
    0,
    num_train - context_length,
    len(data) - num_test - context_length,
]
border2s = [num_train, num_train + num_valid, len(data)]

train_start_index = border1s[0]  # None indicates beginning of dataset
train_end_index = border2s[0]

valid_start_index = train_end_index + context_length
valid_end_index = border2s[1]

test_start_index = valid_end_index + context_length
test_end_index = border2s[2]

train_data = select_by_index(
    data,
    id_columns=id_columns,
    start_index=train_start_index,
    end_index=train_end_index,
)

valid_data = select_by_index(
    data,
    id_columns=id_columns,
    start_index=valid_start_index,
    end_index=valid_end_index,
)
test_data = select_by_index(
    data,
    id_columns=id_columns,
    start_index=test_start_index,
    end_index=test_end_index,
)

tsp = TimeSeriesPreprocessor(
    timestamp_column=timestamp_column,
    id_columns=id_columns,
    target_columns=forecast_columns,
    scaling=True,
)
tsp = tsp.train(train_data)

In [ ]:
print("Training Data Range: {} to {}".format(train_start_index, train_end_index))
print("Validation Data Range: {} to {}".format(valid_start_index, valid_end_index))
print("Testing Data Range: {} to {}".format(test_start_index, test_end_index))


Training Data Range: 0 to 11011
Validation Data Range: 11043 to 12584
Testing Data Range: 12616 to 15730


In [ ]:
train_dataset = ForecastDFDataset(
    tsp.preprocess(train_data),
    id_columns=id_columns,
    timestamp_column="date",
    target_columns=forecast_columns,
    context_length=context_length,
    prediction_length=forecast_horizon,
)
valid_dataset = ForecastDFDataset(
    tsp.preprocess(valid_data),
    id_columns=id_columns,
    timestamp_column="date",
    target_columns=forecast_columns,
    context_length=context_length,
    prediction_length=forecast_horizon,
)
test_dataset = ForecastDFDataset(
    tsp.preprocess(test_data),
    id_columns=id_columns,
    timestamp_column="date",
    target_columns=forecast_columns,
    context_length=context_length,
    prediction_length=forecast_horizon,
)

# Configure the Model

In [ ]:
config = PatchTSTConfig(
    num_input_channels=len(forecast_columns),
    context_length=context_length,
    patch_length=patch_length,
    patch_stride=patch_length,
    prediction_length=forecast_horizon,
    random_mask_ratio=0.4,
    d_model=16,
    num_attention_heads=8,
    num_hidden_layers=3,
    ffn_dim=256,
    dropout=0.2,
    head_dropout=0.2,
    pooling_type=None,
    channel_attention=False,
    scaling="std",
    loss="mse",
    pre_norm=True,
    norm_type="batchnorm",
)
model = PatchTSTForPrediction(config)

# Configure Hyperparameter Tuning

In [ ]:
def optuna_hp_space(trial: optuna.Trial):
    return {
        "learning_rate": trial.suggest_loguniform("learning_rate", 1e-8, 1e-2),  # Granular range
        "per_device_train_batch_size": trial.suggest_categorical("per_device_train_batch_size", [16, 32, 64, 128]),
        "num_train_epochs": trial.suggest_int("num_train_epochs", 50, 300, step=20),
        "dataloader_num_workers": trial.suggest_int("dataloader_num_workers", 0, 16, step=4),
        "weight_decay": trial.suggest_float("weight_decay", 0.0, 0.3, step=0.05),  # Granular steps for weight decay
        "per_device_eval_batch_size": trial.suggest_categorical("per_device_eval_batch_size", [16, 32, 64, 128]),
    }


In [ ]:
def model_init(trial):
    return PatchTSTForPrediction(config)

In [ ]:
training_args = TrainingArguments(
    output_dir="./checkpoint/output_dir",
    overwrite_output_dir=True,
    do_eval=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    save_total_limit=3,
    logging_dir="./checkpoint/logging_dir",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    num_train_epochs=200,  # Note: The actual number of epochs might be lower due to early stopping
    label_names=["future_values"],
)

trainer = Trainer(
    model=None,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    model_init=model_init,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=30, early_stopping_threshold=0.00001)]
)

# Start the hyperparameter search
best_run = trainer.hyperparameter_search(
    backend="optuna",
    n_trials=30,
    direction="minimize",
)

print("Best run:", best_run)


[I 2026-05-17 11:31:57,690] A new study created in memory with name: no-name-1fcde62f-d385-4722-9715-4cd7b09d01a2
/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 3


wandb: You chose "Don't visualize my results"
wandb: Using W&B in offline mode.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss
1,0.011300,0.042977
2,0.011200,0.042594


[I 2026-05-17 11:33:53,903] Trial 0 finished with value: 0.04259368032217026 and parameters: {'learning_rate': 3.385468276978239e-06, 'num_train_epochs': 2, 'seed': 18, 'per_device_train_batch_size': 4}. Best is trial 0 with value: 0.04259368032217026.


...

[I 2026-05-17 11:52:41,077] Trial 28 pruned. 


eval/loss,▁
eval/runtime,▁
eval/samples_per_second,▁
eval/steps_per_second,▁
train/epoch,▁▁
train/global_step,▁▁
train/grad_norm,▁
train/learning_rate,▁
train/loss,▁
eval/loss,0.01929
eval/runtime,1.2001


Epoch,Training Loss,Validation Loss
1,0.011200,0.041979
2,0.010800,0.040804


[I 2026-05-17 11:54:05,436] Trial 29 finished with value: 0.04080433025956154 and parameters: {'learning_rate': 5.662504026158526e-06, 'num_train_epochs': 2, 'seed': 18, 'per_device_train_batch_size': 4}. Best is trial 23 with value: 0.018153175711631775.


Best run: BestRun(run_id='23', objective=0.018153175711631775, hyperparameters={'learning_rate': 9.918344393260846e-05, 'num_train_epochs': 3, 'seed': 31, 'per_device_train_batch_size': 8}, run_summary=None)


# Train Model on Best Hyperparameters

In [ ]:

best_hyperparameters = best_run.hyperparameters

# Update training arguments with the best hyperparameters
training_args = TrainingArguments(
    output_dir="./checkpoint/output_dir",
    overwrite_output_dir=True,
    learning_rate=best_hyperparameters['learning_rate'],
    per_device_train_batch_size=int(best_hyperparameters['per_device_train_batch_size']),  # Make sure to cast to int if needed
    do_eval=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    save_total_limit=3,
    logging_dir="./checkpoint/logging_dir",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    num_train_epochs=200,  # This can be adjusted based on your previous experience
    label_names=["future_values"],
)


In [ ]:
# Reinitialize the Trainer with the updated arguments
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=30, early_stopping_threshold=0.00001)]
)

# Train the model with the best hyperparameters
trainer.train()


Epoch,Training Loss,Validation Loss
1,0.005500,0.018982
2,0.004100,0.017923
3,0.004100,0.018368
4,0.004000,0.017855
5,0.004100,0.018288
6,0.004100,0.018767
7,0.004000,0.017921
8,0.004000,0.018542
9,0.004000,0.018485
10,0.003900,0.018324


TrainOutput(global_step=46648, training_loss=0.004022827362691205, metrics={'train_runtime': 775.2822, 'train_samples_per_second': 2829.937, 'train_steps_per_second': 353.936, 'total_flos': 2127310824960.0, 'train_loss': 0.004022827362691205, 'epoch': 34.0})

# Display results

In [ ]:
results_valid_dataset = trainer.evaluate(valid_dataset)
print("Valid Results:", results_valid_dataset)
results_test_dataset = trainer.evaluate(test_dataset)
print("Test Results:", results_test_dataset)


Valid Results: {'eval_loss': 0.017855091020464897, 'eval_runtime': 1.4605, 'eval_samples_per_second': 1027.06, 'eval_steps_per_second': 128.725, 'epoch': 34.0}
Test Results: {'eval_loss': 0.047965727746486664, 'eval_runtime': 3.3173, 'eval_samples_per_second': 926.357, 'eval_steps_per_second': 116.058, 'epoch': 34.0}


In [ ]:
results_valid_dataset

{'eval_loss': 0.017855091020464897,
 'eval_runtime': 1.4605,
 'eval_samples_per_second': 1027.06,
 'eval_steps_per_second': 128.725,
 'epoch': 34.0}